In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("explore").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

In [3]:
df = spark.read.csv("data/feature_clickstream.csv", header=True, inferSchema=True)
df.printSchema()
df.show(5)

root
 |-- fe_1: integer (nullable = true)
 |-- fe_2: integer (nullable = true)
 |-- fe_3: integer (nullable = true)
 |-- fe_4: integer (nullable = true)
 |-- fe_5: integer (nullable = true)
 |-- fe_6: integer (nullable = true)
 |-- fe_7: integer (nullable = true)
 |-- fe_8: integer (nullable = true)
 |-- fe_9: integer (nullable = true)
 |-- fe_10: integer (nullable = true)
 |-- fe_11: integer (nullable = true)
 |-- fe_12: integer (nullable = true)
 |-- fe_13: integer (nullable = true)
 |-- fe_14: integer (nullable = true)
 |-- fe_15: integer (nullable = true)
 |-- fe_16: integer (nullable = true)
 |-- fe_17: integer (nullable = true)
 |-- fe_18: integer (nullable = true)
 |-- fe_19: integer (nullable = true)
 |-- fe_20: integer (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- snapshot_date: date (nullable = true)

+----+----+----+----+----+----+----+----+----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----------+-------------+
|fe_1|fe_2|fe_3|fe

In [4]:
df2 = spark.read.csv("data/lms_loan_daily.csv", header=True, inferSchema=True)
df2.printSchema()
df2.show(5)

root
 |-- loan_id: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- loan_start_date: date (nullable = true)
 |-- tenure: integer (nullable = true)
 |-- installment_num: integer (nullable = true)
 |-- loan_amt: integer (nullable = true)
 |-- due_amt: double (nullable = true)
 |-- paid_amt: double (nullable = true)
 |-- overdue_amt: double (nullable = true)
 |-- balance: double (nullable = true)
 |-- snapshot_date: date (nullable = true)

+--------------------+-----------+---------------+------+---------------+--------+-------+--------+-----------+-------+-------------+
|             loan_id|Customer_ID|loan_start_date|tenure|installment_num|loan_amt|due_amt|paid_amt|overdue_amt|balance|snapshot_date|
+--------------------+-----------+---------------+------+---------------+--------+-------+--------+-----------+-------+-------------+
|CUS_0x1000_2023_0...| CUS_0x1000|     2023-05-01|    10|              0|   10000|    0.0|     0.0|        0.0|10000.0|   2023-05-01|

In [5]:
df3 = spark.read.csv("data/features_financials.csv", header=True, inferSchema=True)
df3.printSchema()
df3.show(5)

root
 |-- Customer_ID: string (nullable = true)
 |-- Annual_Income: string (nullable = true)
 |-- Monthly_Inhand_Salary: double (nullable = true)
 |-- Num_Bank_Accounts: integer (nullable = true)
 |-- Num_Credit_Card: integer (nullable = true)
 |-- Interest_Rate: integer (nullable = true)
 |-- Num_of_Loan: string (nullable = true)
 |-- Type_of_Loan: string (nullable = true)
 |-- Delay_from_due_date: integer (nullable = true)
 |-- Num_of_Delayed_Payment: string (nullable = true)
 |-- Changed_Credit_Limit: string (nullable = true)
 |-- Num_Credit_Inquiries: double (nullable = true)
 |-- Credit_Mix: string (nullable = true)
 |-- Outstanding_Debt: string (nullable = true)
 |-- Credit_Utilization_Ratio: double (nullable = true)
 |-- Credit_History_Age: string (nullable = true)
 |-- Payment_of_Min_Amount: string (nullable = true)
 |-- Total_EMI_per_month: double (nullable = true)
 |-- Amount_invested_monthly: string (nullable = true)
 |-- Payment_Behaviour: string (nullable = true)
 |-- Mont

In [6]:
df4 = spark.read.csv("data/features_attributes.csv", header=True, inferSchema=True)
df4.printSchema()
df4.show(5)

root
 |-- Customer_ID: string (nullable = true)
 |-- Name: string (nullable = true)
 |-- Age: string (nullable = true)
 |-- SSN: string (nullable = true)
 |-- Occupation: string (nullable = true)
 |-- snapshot_date: date (nullable = true)

+-----------+--------------+---+-----------+-------------+-------------+
|Customer_ID|          Name|Age|        SSN|   Occupation|snapshot_date|
+-----------+--------------+---+-----------+-------------+-------------+
| CUS_0x1000|Alistair Barrf| 18|913-74-1218|       Lawyer|   2023-05-01|
| CUS_0x1009|        Arunah| 26|063-67-6938|     Mechanic|   2025-01-01|
| CUS_0x100b|      Shirboni| 19|  #F%$D@*&8|Media_Manager|   2024-03-01|
| CUS_0x1011|     Schneyerh| 44|793-05-8223|       Doctor|   2023-11-01|
| CUS_0x1013|      Cameront| 44|930-49-9615|     Mechanic|   2023-12-01|
+-----------+--------------+---+-----------+-------------+-------------+
only showing top 5 rows



In [7]:
for name, d in [("clickstream", df), ("loan_daily", df2), ("financials", df3), ("attributes", df4)]:
    dates = sorted(r["snapshot_date"] for r in d.select("snapshot_date").distinct().collect())
    print(f"{name:12s} rows={d.count():>7d}  snapshot_dates={len(dates):>3d}  range=({dates[0]} -> {dates[-1]})")
    

clickstream  rows= 215376  snapshot_dates= 24  range=(2023-01-01 -> 2024-12-01)
loan_daily   rows= 137500  snapshot_dates= 35  range=(2023-01-01 -> 2025-11-01)
financials   rows=  12500  snapshot_dates= 25  range=(2023-01-01 -> 2025-01-01)
attributes   rows=  12500  snapshot_dates= 25  range=(2023-01-01 -> 2025-01-01)


In [8]:
print("financials distinct customers:", df3.select("Customer_ID").distinct().count())
print("attributes distinct customers:", df4.select("Customer_ID").distinct().count())
print("clickstream distinct customers:", df.select("Customer_ID").distinct().count())
print("loan_daily distinct customers:", df2.select("Customer_ID").distinct().count())
print("loan_daily distinct loans:", df2.select("loan_id").distinct().count())

financials distinct customers: 12500
attributes distinct customers: 12500
clickstream distinct customers: 8974
loan_daily distinct customers: 12500
loan_daily distinct loans: 12500


In [9]:
tenure_dist = df2.select("tenure").distinct().orderBy("tenure").collect()
print([r["tenure"] for r in tenure_dist])

max_installment = df2.groupBy("loan_id").agg({"installment_num": "max"}).collect()
import statistics
vals = [r["max(installment_num)"] for r in max_installment]
print("min max_installment:", min(vals), " median:", statistics.median(vals), " max:", max(vals))

[10]
min max_installment: 10  median: 10.0  max: 10


In [11]:
import pyspark.sql.functions as F

check = (
    df2.select("Customer_ID", "loan_start_date")
       .distinct()
       .join(df3.select("Customer_ID", "snapshot_date"), on="Customer_ID", how="inner")
       .withColumn("dates_match", F.col("loan_start_date") == F.col("snapshot_date"))
)
check.groupBy("dates_match").count().show()

+-----------+-----+
|dates_match|count|
+-----------+-----+
|       true|12500|
+-----------+-----+



In [12]:
check2 = (
    df2.select("Customer_ID", "loan_start_date")
       .distinct()
       .join(df.select("Customer_ID", "snapshot_date"), on="Customer_ID", how="inner")
       .withColumn("dates_match", F.col("loan_start_date") == F.col("snapshot_date"))
)
check2.groupBy("dates_match").count().show()

+-----------+------+
|dates_match| count|
+-----------+------+
|       true|  8974|
|      false|206402|
+-----------+------+



In [13]:
import os

BASE_DIR = os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, "data")
DATAMART_DIR = os.path.join(BASE_DIR, "datamart")
BRONZE_DIR = os.path.join(DATAMART_DIR, "bronze")
SILVER_DIR = os.path.join(DATAMART_DIR, "silver")
GOLD_DIR = os.path.join(DATAMART_DIR, "gold")

os.makedirs(BRONZE_DIR, exist_ok=True)
os.makedirs(SILVER_DIR, exist_ok=True)
os.makedirs(GOLD_DIR, exist_ok=True)

In [14]:
RAW_FILE_MAP = {
    "clickstream": "feature_clickstream.csv",
    "loan_daily": "lms_loan_daily.csv",
    "financials": "features_financials.csv",
    "attributes": "features_attributes.csv",
}

def process_bronze_table(table_name, snapshot_date, spark):
    raw_path = os.path.join(DATA_DIR, RAW_FILE_MAP[table_name])
    df = spark.read.csv(raw_path, header=True, inferSchema=True)
    df = df.filter(df.snapshot_date == snapshot_date)

    out_dir = os.path.join(BRONZE_DIR, table_name)
    os.makedirs(out_dir, exist_ok=True)
    out_name = f"bronze_{table_name}_{snapshot_date.replace('-', '_')}.csv"
    out_path = os.path.join(out_dir, out_name)

    n = df.count()
    df.toPandas().to_csv(out_path, index=False)
    print(f"[bronze] {table_name:12s} {snapshot_date}: {n:>6d} rows -> {out_path}")
    return out_path

In [15]:
process_bronze_table("attributes", "2023-05-01", spark)

[bronze] attributes   2023-05-01:    521 rows -> /home/jovyan/work/datamart/bronze/attributes/bronze_attributes_2023_05_01.csv


'/home/jovyan/work/datamart/bronze/attributes/bronze_attributes_2023_05_01.csv'

In [16]:
def get_snapshot_dates(table_name, spark):
    raw_path = os.path.join(DATA_DIR, RAW_FILE_MAP[table_name])
    df = spark.read.csv(raw_path, header=True, inferSchema=True)
    dates = [r["snapshot_date"] for r in df.select("snapshot_date").distinct().collect()]
    return sorted(str(d) for d in dates)

bronze_paths = {}
for table_name in RAW_FILE_MAP:
    dates = get_snapshot_dates(table_name, spark)
    print(f"\n=== {table_name}: {len(dates)} snapshot dates ===")
    bronze_paths[table_name] = {}
    for d in dates:
        out_path = process_bronze_table(table_name, d, spark)
        bronze_paths[table_name][d] = out_path


=== clickstream: 24 snapshot dates ===
[bronze] clickstream  2023-01-01:   8974 rows -> /home/jovyan/work/datamart/bronze/clickstream/bronze_clickstream_2023_01_01.csv
[bronze] clickstream  2023-02-01:   8974 rows -> /home/jovyan/work/datamart/bronze/clickstream/bronze_clickstream_2023_02_01.csv
[bronze] clickstream  2023-03-01:   8974 rows -> /home/jovyan/work/datamart/bronze/clickstream/bronze_clickstream_2023_03_01.csv
[bronze] clickstream  2023-04-01:   8974 rows -> /home/jovyan/work/datamart/bronze/clickstream/bronze_clickstream_2023_04_01.csv
[bronze] clickstream  2023-05-01:   8974 rows -> /home/jovyan/work/datamart/bronze/clickstream/bronze_clickstream_2023_05_01.csv
[bronze] clickstream  2023-06-01:   8974 rows -> /home/jovyan/work/datamart/bronze/clickstream/bronze_clickstream_2023_06_01.csv
[bronze] clickstream  2023-07-01:   8974 rows -> /home/jovyan/work/datamart/bronze/clickstream/bronze_clickstream_2023_07_01.csv
[bronze] clickstream  2023-08-01:   8974 rows -> /home/jo

In [17]:
df3.select("Annual_Income").distinct().orderBy(F.rand()).show(10, truncate=False)
df3.select("Num_of_Loan").distinct().show(10, truncate=False)
df3.select("Credit_Mix").distinct().show()
df3.select("Payment_of_Min_Amount").distinct().show()
df4.select("Age").distinct().orderBy(F.rand()).show(10, truncate=False)
df4.select("Occupation").distinct().show(20, truncate=False)

+-----------------+
|Annual_Income    |
+-----------------+
|55732.68000000001|
|19125.48         |
|31969.22         |
|35952.45         |
|143552.0         |
|36549.68         |
|19041.48         |
|14639.215        |
|24418.33         |
|54876.2          |
+-----------------+
only showing top 10 rows

+-----------+
|Num_of_Loan|
+-----------+
|1159       |
|7          |
|1_         |
|1171_      |
|1412       |
|387        |
|87         |
|3          |
|966        |
|365        |
+-----------+
only showing top 10 rows

+----------+
|Credit_Mix|
+----------+
|         _|
|      Good|
|       Bad|
|  Standard|
+----------+

+---------------------+
|Payment_of_Min_Amount|
+---------------------+
|                   NM|
|                   No|
|                  Yes|
+---------------------+

+-----+
|Age  |
+-----+
|3986 |
|26_  |
|51   |
|1070_|
|6280 |
|15_  |
|35   |
|1990 |
|6043 |
|813  |
+-----+
only showing top 10 rows

+-------------+
|Occupation   |
+-------------+
|Scientist  

In [18]:
from pyspark.sql.types import DoubleType, IntegerType

def _write_silver(df, table_name, snapshot_date):
    out_dir = os.path.join(SILVER_DIR, table_name)
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, f"silver_{table_name}_{snapshot_date.replace('-', '_')}.parquet")
    df.write.mode("overwrite").parquet(out_path)
    print(f"[silver] {table_name:12s} {snapshot_date}: {df.count():>6d} rows -> {out_path}")
    return df

def _strip_to_number(colname):
    return F.regexp_extract(F.col(colname).cast("string"), r"(-?\d+\.?\d*)", 1)

def process_silver_attributes(bronze_df, snapshot_date):
    df = bronze_df
    df = df.withColumn("Age", _strip_to_number("Age").cast(IntegerType()))
    df = df.withColumn("Age", F.when((F.col("Age") < 18) | (F.col("Age") > 100), None).otherwise(F.col("Age")))
    df = df.withColumn("Occupation", F.when(F.col("Occupation").rlike(r"^[A-Za-z_ ]+$") & (F.col("Occupation") != "_______"), F.col("Occupation")).otherwise(None))
    return _write_silver(df, "attributes", snapshot_date)

def process_silver_financials(bronze_df, snapshot_date):
    df = bronze_df
    df = df.withColumn("Annual_Income", _strip_to_number("Annual_Income").cast(DoubleType()))
    df = df.withColumn("Num_of_Loan", _strip_to_number("Num_of_Loan").cast(IntegerType()))
    df = df.withColumn("Num_of_Loan", F.when((F.col("Num_of_Loan") < 0) | (F.col("Num_of_Loan") > 20), None).otherwise(F.col("Num_of_Loan")))
    df = df.withColumn("Credit_Mix", F.when(F.trim(F.col("Credit_Mix")) == "_", None).otherwise(F.col("Credit_Mix")))
    df = df.withColumn("Payment_of_Min_Amount", F.when(F.col("Payment_of_Min_Amount") == "NM", None).otherwise(F.col("Payment_of_Min_Amount")))
    return _write_silver(df, "financials", snapshot_date)

In [20]:
bronze_attr_path = bronze_paths["attributes"]["2023-05-01"]
bronze_attr_df = spark.read.csv(bronze_attr_path, header=True, inferSchema=True)
silver_attr = process_silver_attributes(bronze_attr_df, "2023-05-01")
silver_attr.show(5)

bronze_fin_path = bronze_paths["financials"]["2023-05-01"]
bronze_fin_df = spark.read.csv(bronze_fin_path, header=True, inferSchema=True)
silver_fin = process_silver_financials(bronze_fin_df, "2023-05-01")
silver_fin.select("Customer_ID", "Annual_Income", "Num_of_Loan", "Credit_Mix", "Payment_of_Min_Amount").show(5)

[silver] attributes   2023-05-01:    521 rows -> /home/jovyan/work/datamart/silver/attributes/silver_attributes_2023_05_01.parquet
+-----------+-----------------+----+-----------+-------------+-------------+
|Customer_ID|             Name| Age|        SSN|   Occupation|snapshot_date|
+-----------+-----------------+----+-----------+-------------+-------------+
| CUS_0x1000|   Alistair Barrf|  18|913-74-1218|       Lawyer|   2023-05-01|
| CUS_0x108a|          Silvial|  38|394-00-8142|   Journalist|   2023-05-01|
| CUS_0x10f9|Gilles Guillaumev|  54|746-79-6875|         NULL|   2023-05-01|
| CUS_0x1119|         Carlozoy|  36|596-61-5235| Entrepreneur|   2023-05-01|
| CUS_0x1192|     Paul Carrell|NULL|880-23-2963|Media_Manager|   2023-05-01|
+-----------+-----------------+----+-----------+-------------+-------------+
only showing top 5 rows

[silver] financials   2023-05-01:    521 rows -> /home/jovyan/work/datamart/silver/financials/silver_financials_2023_05_01.parquet
+-----------+-------

In [22]:
def process_silver_loan_daily(bronze_df, snapshot_date):
    df = bronze_df
    df = (df.withColumn("loan_id", F.col("loan_id").cast("string"))
            .withColumn("Customer_ID", F.col("Customer_ID").cast("string"))
            .withColumn("loan_start_date", F.to_date("loan_start_date"))
            .withColumn("snapshot_date", F.to_date("snapshot_date"))
            .withColumn("tenure", F.col("tenure").cast(IntegerType()))
            .withColumn("installment_num", F.col("installment_num").cast(IntegerType()))
            .withColumn("overdue_amt", F.col("overdue_amt").cast(DoubleType())))
    for c in ["loan_amt", "due_amt", "paid_amt", "overdue_amt", "balance"]:
        df = df.withColumn(c, F.when(F.col(c) < 0, None).otherwise(F.col(c)))
    return _write_silver(df, "loan_daily", snapshot_date)

def process_silver_clickstream(bronze_df, snapshot_date):
    df = bronze_df.withColumn("Customer_ID", F.col("Customer_ID").cast("string"))
    df = df.withColumn("snapshot_date", F.to_date("snapshot_date"))
    return _write_silver(df, "clickstream", snapshot_date)

SILVER_PROCESSORS = {
    "attributes": process_silver_attributes,
    "financials": process_silver_financials,
    "loan_daily": process_silver_loan_daily,
    "clickstream": process_silver_clickstream,
}

for table_name, paths_by_date in bronze_paths.items():
    fn = SILVER_PROCESSORS[table_name]
    for d, bronze_path in paths_by_date.items():
        bronze_df = spark.read.csv(bronze_path, header=True, inferSchema=True)
        fn(bronze_df, d)

[silver] clickstream  2023-01-01:   8974 rows -> /home/jovyan/work/datamart/silver/clickstream/silver_clickstream_2023_01_01.parquet
[silver] clickstream  2023-02-01:   8974 rows -> /home/jovyan/work/datamart/silver/clickstream/silver_clickstream_2023_02_01.parquet
[silver] clickstream  2023-03-01:   8974 rows -> /home/jovyan/work/datamart/silver/clickstream/silver_clickstream_2023_03_01.parquet
[silver] clickstream  2023-04-01:   8974 rows -> /home/jovyan/work/datamart/silver/clickstream/silver_clickstream_2023_04_01.parquet
[silver] clickstream  2023-05-01:   8974 rows -> /home/jovyan/work/datamart/silver/clickstream/silver_clickstream_2023_05_01.parquet
[silver] clickstream  2023-06-01:   8974 rows -> /home/jovyan/work/datamart/silver/clickstream/silver_clickstream_2023_06_01.parquet
[silver] clickstream  2023-07-01:   8974 rows -> /home/jovyan/work/datamart/silver/clickstream/silver_clickstream_2023_07_01.parquet
[silver] clickstream  2023-08-01:   8974 rows -> /home/jovyan/work/da

In [23]:
silver_loan_daily = spark.read.parquet(os.path.join(SILVER_DIR, "loan_daily", "*.parquet"))

MOB_THRESHOLD = 6

label_store = (
    silver_loan_daily
    .filter(F.col("installment_num") == MOB_THRESHOLD)
    .withColumn("label", F.when(F.col("overdue_amt") > 0, 1).otherwise(0))
    .select("loan_id", "Customer_ID", "loan_start_date", "label")
)

print("label_store rows:", label_store.count())
label_store.groupBy("label").count().show()
label_store.show(5)

label_store rows: 12500
+-----+-----+
|label|count|
+-----+-----+
|    1| 3602|
|    0| 8898|
+-----+-----+

+--------------------+-----------+---------------+-----+
|             loan_id|Customer_ID|loan_start_date|label|
+--------------------+-----------+---------------+-----+
|CUS_0x1108_2024_0...| CUS_0x1108|     2024-07-01|    1|
|CUS_0x13b5_2024_0...| CUS_0x13b5|     2024-07-01|    0|
|CUS_0x13b8_2024_0...| CUS_0x13b8|     2024-07-01|    0|
|CUS_0x144f_2024_0...| CUS_0x144f|     2024-07-01|    1|
|CUS_0x14ec_2024_0...| CUS_0x14ec|     2024-07-01|    1|
+--------------------+-----------+---------------+-----+
only showing top 5 rows



In [24]:
label_out_dir = os.path.join(GOLD_DIR, "label_store")
os.makedirs(label_out_dir, exist_ok=True)
label_store.write.mode("overwrite").parquet(os.path.join(label_out_dir, "gold_label_store.parquet"))
print("label store written.")

label store written.


In [25]:
silver_attributes = spark.read.parquet(os.path.join(SILVER_DIR, "attributes", "*.parquet"))
silver_financials = spark.read.parquet(os.path.join(SILVER_DIR, "financials", "*.parquet"))
silver_clickstream = spark.read.parquet(os.path.join(SILVER_DIR, "clickstream", "*.parquet"))

feature_store = (
    silver_attributes.alias("a")
    .join(
        silver_financials.alias("f"),
        on=["Customer_ID", "snapshot_date"],
        how="inner",
    )
    .drop(F.col("f.snapshot_date"))
)

# clickstream: only keep rows where its own snapshot_date matches the
# customer's loan-origination date (= attributes/financials snapshot_date),
# then left-join so customers with no clickstream coverage keep null features
click_at_origination = silver_clickstream.withColumnRenamed("snapshot_date", "click_snapshot_date")

feature_store = feature_store.join(
    click_at_origination,
    on=(feature_store["Customer_ID"] == click_at_origination["Customer_ID"]) &
       (feature_store["snapshot_date"] == click_at_origination["click_snapshot_date"]),
    how="left",
).drop(click_at_origination["Customer_ID"]).drop("click_snapshot_date")

print("feature_store rows:", feature_store.count())
feature_store.select("Customer_ID", "snapshot_date", "Age", "Occupation", "Annual_Income", "fe_1").show(5)

feature_store rows: 12500
+-----------+-------------+---+------------+-------------+----+
|Customer_ID|snapshot_date|Age|  Occupation|Annual_Income|fe_1|
+-----------+-------------+---+------------+-------------+----+
| CUS_0x10ac|   2024-08-01| 29|   Developer|    16718.645|NULL|
| CUS_0x10c5|   2024-08-01| 24|        NULL|    158623.16|NULL|
| CUS_0x1145|   2024-08-01| 24|     Teacher|     17975.32|NULL|
| CUS_0x11ac|   2024-08-01| 26|  Journalist|     69136.54|NULL|
| CUS_0x122c|   2024-08-01| 48|Entrepreneur|     36346.13|NULL|
+-----------+-------------+---+------------+-------------+----+
only showing top 5 rows



In [26]:
feature_store.filter(F.col("fe_1").isNotNull()).count()

8974

In [28]:
NUMERIC_FINANCIAL_COLS = [
    "Annual_Income", "Monthly_Inhand_Salary", "Num_Bank_Accounts", "Num_Credit_Card",
    "Interest_Rate", "Num_of_Loan", "Delay_from_due_date", "Num_of_Delayed_Payment",
    "Changed_Credit_Limit", "Num_Credit_Inquiries", "Outstanding_Debt",
    "Credit_Utilization_Ratio", "Total_EMI_per_month", "Amount_invested_monthly",
    "Monthly_Balance",
]

def process_silver_financials(bronze_df, snapshot_date):
    df = bronze_df
    for c in NUMERIC_FINANCIAL_COLS:
        df = df.withColumn(c, _strip_to_number(c).cast(DoubleType()))
    df = df.withColumn("Num_of_Loan", F.when((F.col("Num_of_Loan") < 0) | (F.col("Num_of_Loan") > 20), None).otherwise(F.col("Num_of_Loan")).cast(IntegerType()))
    df = df.withColumn("Credit_Mix", F.when(F.trim(F.col("Credit_Mix")) == "_", None).otherwise(F.col("Credit_Mix")))
    df = df.withColumn("Payment_of_Min_Amount", F.when(F.col("Payment_of_Min_Amount") == "NM", None).otherwise(F.col("Payment_of_Min_Amount")))
    return _write_silver(df, "financials", snapshot_date)

In [29]:
for d, bronze_path in bronze_paths["financials"].items():
    bronze_df = spark.read.csv(bronze_path, header=True, inferSchema=True)
    process_silver_financials(bronze_df, d)

[silver] financials   2023-01-01:    530 rows -> /home/jovyan/work/datamart/silver/financials/silver_financials_2023_01_01.parquet
[silver] financials   2023-02-01:    501 rows -> /home/jovyan/work/datamart/silver/financials/silver_financials_2023_02_01.parquet
[silver] financials   2023-03-01:    506 rows -> /home/jovyan/work/datamart/silver/financials/silver_financials_2023_03_01.parquet
[silver] financials   2023-04-01:    510 rows -> /home/jovyan/work/datamart/silver/financials/silver_financials_2023_04_01.parquet
[silver] financials   2023-05-01:    521 rows -> /home/jovyan/work/datamart/silver/financials/silver_financials_2023_05_01.parquet
[silver] financials   2023-06-01:    517 rows -> /home/jovyan/work/datamart/silver/financials/silver_financials_2023_06_01.parquet
[silver] financials   2023-07-01:    471 rows -> /home/jovyan/work/datamart/silver/financials/silver_financials_2023_07_01.parquet
[silver] financials   2023-08-01:    481 rows -> /home/jovyan/work/datamart/silver/

In [31]:
silver_attributes = spark.read.parquet(os.path.join(SILVER_DIR, "attributes", "*.parquet"))
silver_financials = spark.read.parquet(os.path.join(SILVER_DIR, "financials", "*.parquet"))
silver_clickstream = spark.read.parquet(os.path.join(SILVER_DIR, "clickstream", "*.parquet"))

feature_store = (
    silver_attributes.alias("a")
    .join(silver_financials.alias("f"), on=["Customer_ID", "snapshot_date"], how="inner")
    .drop(F.col("f.snapshot_date"))
)

click_at_origination = silver_clickstream.withColumnRenamed("snapshot_date", "click_snapshot_date")
feature_store = feature_store.join(
    click_at_origination,
    on=(feature_store["Customer_ID"] == click_at_origination["Customer_ID"]) &
       (feature_store["snapshot_date"] == click_at_origination["click_snapshot_date"]),
    how="left",
).drop(click_at_origination["Customer_ID"]).drop("click_snapshot_date")

feat_out_dir = os.path.join(GOLD_DIR, "feature_store")
os.makedirs(feat_out_dir, exist_ok=True)
feature_store.write.mode("overwrite").parquet(os.path.join(feat_out_dir, "gold_feature_store.parquet"))
print("feature store written, rows:", feature_store.count())

feature store written, rows: 12500


In [32]:
joined = label_store.join(
    feature_store,
    on=(label_store["Customer_ID"] == feature_store["Customer_ID"]) &
       (label_store["loan_start_date"] == feature_store["snapshot_date"]),
    how="inner",
)
print("joined rows:", joined.count())

joined rows: 12500


In [33]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

pdf = joined.toPandas()
y = pdf["label"].values
drop_cols = ["loan_id", "Customer_ID", "loan_start_date", "snapshot_date", "label"]
X = pdf.drop(columns=[c for c in drop_cols if c in pdf.columns])

cat_cols = [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]
num_cols = [c for c in X.columns if c not in cat_cols]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

num_imputer = SimpleImputer(strategy="median")
X_train_num = num_imputer.fit_transform(X_train[num_cols])
X_test_num = num_imputer.transform(X_test[num_cols])

cat_imputer = SimpleImputer(strategy="constant", fill_value="Unknown")
X_train_cat_raw = cat_imputer.fit_transform(X_train[cat_cols])
X_test_cat_raw = cat_imputer.transform(X_test[cat_cols])
enc = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
X_train_cat = enc.fit_transform(X_train_cat_raw)
X_test_cat = enc.transform(X_test_cat_raw)

X_train_all = np.hstack([X_train_num, X_train_cat])
X_test_all = np.hstack([X_test_num, X_test_cat])

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_all, y_train)
auc = roc_auc_score(y_test, clf.predict_proba(X_test_all)[:, 1])
print(f"sanity-check test AUC: {auc:.3f}")

sanity-check test AUC: 0.500


In [35]:
"""
Bronze layer processing.

Bronze = raw landing zone. Preserves source records at snapshot level --
no business cleaning, no type coercion, no feature engineering -- and
persists them as partitioned CSV snapshots, one partition per snapshot_date.
"""
import os
from pyspark.sql import SparkSession

RAW_FILE_MAP = {
    "clickstream": "feature_clickstream.csv",
    "loan_daily": "lms_loan_daily.csv",
    "financials": "features_financials.csv",
    "attributes": "features_attributes.csv",
}


def process_bronze_table(table_name: str, snapshot_date: str, data_dir: str,
                          bronze_dir: str, spark: SparkSession) -> str:
    """Read the raw CSV for `table_name`, filter to rows for `snapshot_date`,
    and write out a bronze snapshot partition unchanged. Returns the file
    path (not the dataframe), forcing the caller to re-read from disk for
    the silver step -- bronze must be a real persisted landing layer that
    silver is built FROM, not a side-effect write next to an in-memory
    dataframe silver happens to reuse."""
    raw_filename = RAW_FILE_MAP[table_name]
    raw_path = os.path.join(data_dir, raw_filename)

    df = spark.read.csv(raw_path, header=True, inferSchema=True)
    df = df.filter(df.snapshot_date == snapshot_date)

    out_dir = os.path.join(bronze_dir, table_name)
    os.makedirs(out_dir, exist_ok=True)
    out_name = f"bronze_{table_name}_{snapshot_date.replace('-', '_')}.csv"
    out_path = os.path.join(out_dir, out_name)

    n = df.count()
    df.toPandas().to_csv(out_path, index=False)
    print(f"[bronze] {table_name:12s} {snapshot_date}: {n:>6d} rows -> {out_path}")
    return out_path

In [36]:
"""
Silver layer processing.

Silver = cleaned, conformed, typed data at the same grain as bronze, with
malformed/placeholder values fixed or nulled out.
"""
import os
import pyspark.sql.functions as F
from pyspark.sql.types import DoubleType, IntegerType


def _write_silver(df, table_name, snapshot_date, silver_dir):
    out_dir = os.path.join(silver_dir, table_name)
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, f"silver_{table_name}_{snapshot_date.replace('-', '_')}.parquet")
    df.write.mode("overwrite").parquet(out_path)
    print(f"[silver] {table_name:12s} {snapshot_date}: {df.count():>6d} rows -> {out_path}")
    return df


def _strip_to_number(colname):
    return F.regexp_extract(F.col(colname).cast("string"), r"(-?\d+\.?\d*)", 1)


def process_silver_attributes(bronze_df, snapshot_date, silver_dir):
    df = bronze_df
    df = df.withColumn("Age", _strip_to_number("Age").cast(IntegerType()))
    df = df.withColumn("Age", F.when((F.col("Age") < 18) | (F.col("Age") > 100), None).otherwise(F.col("Age")))
    df = df.withColumn(
        "Occupation",
        F.when(F.col("Occupation").rlike(r"^[A-Za-z_ ]+$") & (F.col("Occupation") != "_______"), F.col("Occupation")).otherwise(None),
    )
    return _write_silver(df, "attributes", snapshot_date, silver_dir)


NUMERIC_FINANCIAL_COLS = [
    "Annual_Income", "Monthly_Inhand_Salary", "Num_Bank_Accounts", "Num_Credit_Card",
    "Interest_Rate", "Num_of_Loan", "Delay_from_due_date", "Num_of_Delayed_Payment",
    "Changed_Credit_Limit", "Num_Credit_Inquiries", "Outstanding_Debt",
    "Credit_Utilization_Ratio", "Total_EMI_per_month", "Amount_invested_monthly",
    "Monthly_Balance",
]


def process_silver_financials(bronze_df, snapshot_date, silver_dir):
    df = bronze_df
    for c in NUMERIC_FINANCIAL_COLS:
        df = df.withColumn(c, _strip_to_number(c).cast(DoubleType()))
    df = df.withColumn(
        "Num_of_Loan",
        F.when((F.col("Num_of_Loan") < 0) | (F.col("Num_of_Loan") > 20), None).otherwise(F.col("Num_of_Loan")).cast(IntegerType()),
    )
    df = df.withColumn("Credit_Mix", F.when(F.trim(F.col("Credit_Mix")) == "_", None).otherwise(F.col("Credit_Mix")))
    df = df.withColumn("Payment_of_Min_Amount", F.when(F.col("Payment_of_Min_Amount") == "NM", None).otherwise(F.col("Payment_of_Min_Amount")))
    return _write_silver(df, "financials", snapshot_date, silver_dir)


def process_silver_loan_daily(bronze_df, snapshot_date, silver_dir):
    df = bronze_df
    df = (
        df.withColumn("loan_id", F.col("loan_id").cast("string"))
          .withColumn("Customer_ID", F.col("Customer_ID").cast("string"))
          .withColumn("loan_start_date", F.to_date("loan_start_date"))
          .withColumn("snapshot_date", F.to_date("snapshot_date"))
          .withColumn("tenure", F.col("tenure").cast(IntegerType()))
          .withColumn("installment_num", F.col("installment_num").cast(IntegerType()))
          .withColumn("overdue_amt", F.col("overdue_amt").cast(DoubleType()))
    )
    for c in ["loan_amt", "due_amt", "paid_amt", "overdue_amt", "balance"]:
        df = df.withColumn(c, F.when(F.col(c) < 0, None).otherwise(F.col(c)))
    return _write_silver(df, "loan_daily", snapshot_date, silver_dir)


def process_silver_clickstream(bronze_df, snapshot_date, silver_dir):
    df = bronze_df.withColumn("Customer_ID", F.col("Customer_ID").cast("string"))
    df = df.withColumn("snapshot_date", F.to_date("snapshot_date"))
    return _write_silver(df, "clickstream", snapshot_date, silver_dir)


SILVER_PROCESSORS = {
    "attributes": process_silver_attributes,
    "financials": process_silver_financials,
    "loan_daily": process_silver_loan_daily,
    "clickstream": process_silver_clickstream,
}

In [38]:
"""
Gold layer processing.

Gold = business-ready tables: the label store (one row per loan, labeled at
a fixed mob checkpoint) and the feature store (one row per customer, built
from attributes + financials + origination-date clickstream).
"""
import os
import pyspark.sql.functions as F

MOB_THRESHOLD = 6


def process_gold_label_store(silver_loan_daily_all, gold_dir):
    """silver_loan_daily_all: the UNION of all loan_daily silver snapshots
    (read with a wildcard path), not a single monthly partition -- we need
    every loan's row at installment_num == MOB_THRESHOLD, and different
    loans hit that mob at different calendar snapshot_dates."""
    label_store = (
        silver_loan_daily_all
        .filter(F.col("installment_num") == MOB_THRESHOLD)
        .withColumn("label", F.when(F.col("overdue_amt") > 0, 1).otherwise(0))
        .select("loan_id", "Customer_ID", "loan_start_date", "label")
    )

    out_dir = os.path.join(gold_dir, "label_store")
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, "gold_label_store.parquet")
    label_store.write.mode("overwrite").parquet(out_path)
    n = label_store.count()
    bad_rate = label_store.filter(F.col("label") == 1).count() / n
    print(f"[gold] label_store: {n:>6d} rows, bad rate={bad_rate:.3f} -> {out_path}")
    return label_store


def process_gold_feature_store(silver_attributes_all, silver_financials_all,
                                silver_clickstream_all, gold_dir):
    """Join attributes + financials on (Customer_ID, snapshot_date) -- always
    aligned since both are captured at loan origination -- then left-join
    clickstream filtered to rows where its own snapshot_date matches that
    same origination date, so customers with no clickstream coverage keep
    null clickstream features rather than being dropped."""
    feature_store = (
        silver_attributes_all.alias("a")
        .join(silver_financials_all.alias("f"), on=["Customer_ID", "snapshot_date"], how="inner")
        .drop(F.col("f.snapshot_date"))
    )

    click = silver_clickstream_all.withColumnRenamed("snapshot_date", "click_snapshot_date")
    feature_store = feature_store.join(
        click,
        on=(feature_store["Customer_ID"] == click["Customer_ID"]) &
           (feature_store["snapshot_date"] == click["click_snapshot_date"]),
        how="left",
    ).drop(click["Customer_ID"]).drop("click_snapshot_date")

    out_dir = os.path.join(gold_dir, "feature_store")
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, "gold_feature_store.parquet")
    feature_store.write.mode("overwrite").parquet(out_path)
    print(f"[gold] feature_store: {feature_store.count():>6d} rows -> {out_path}")
    return feature_store

In [40]:
import os
print(os.getcwd())
print(os.listdir())
print(os.listdir("utils") if os.path.exists("utils") else "no utils folder found here")

/home/jovyan/work
['.ipynb_checkpoints', 'data', 'datamart', 'docker-compose.yaml', 'Dockerfile', 'exploration.ipynb', 'requirements.txt']
no utils folder found here


In [39]:
"""
main.py -- orchestrates the full bronze -> silver -> gold pipeline for
CS611 Assignment 1 (Data Processing Pipelines).

Usage:
    python main.py

Builds a `datamart/` folder containing:
    datamart/bronze/{clickstream,loan_daily,financials,attributes}/...
    datamart/silver/{clickstream,loan_daily,financials,attributes}/...
    datamart/gold/{label_store,feature_store}/...
"""
import os

from pyspark.sql import SparkSession

from utils.data_processing_bronze import process_bronze_table, RAW_FILE_MAP
from utils.data_processing_silver import SILVER_PROCESSORS
from utils.data_processing_gold import (
    process_gold_label_store,
    process_gold_feature_store,
    MOB_THRESHOLD,
)

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
DATA_DIR = os.path.join(BASE_DIR, "data")
DATAMART_DIR = os.path.join(BASE_DIR, "datamart")
BRONZE_DIR = os.path.join(DATAMART_DIR, "bronze")
SILVER_DIR = os.path.join(DATAMART_DIR, "silver")
GOLD_DIR = os.path.join(DATAMART_DIR, "gold")


def get_snapshot_dates(spark, table_name):
    raw_path = os.path.join(DATA_DIR, RAW_FILE_MAP[table_name])
    df = spark.read.csv(raw_path, header=True, inferSchema=True)
    dates = [r["snapshot_date"] for r in df.select("snapshot_date").distinct().collect()]
    return sorted(str(d) for d in dates)


def run_bronze_silver(spark, table_name):
    dates = get_snapshot_dates(spark, table_name)
    print(f"\n=== {table_name}: {len(dates)} monthly snapshots ({dates[0]} -> {dates[-1]}) ===")
    silver_fn = SILVER_PROCESSORS[table_name]
    for d in dates:
        bronze_path = process_bronze_table(table_name, d, DATA_DIR, BRONZE_DIR, spark)
        bronze_df = spark.read.csv(bronze_path, header=True, inferSchema=True)
        silver_fn(bronze_df, d, SILVER_DIR)


def main():
    spark = SparkSession.builder.appName("cs611_assignment1_pipeline").master("local[*]").getOrCreate()
    spark.sparkContext.setLogLevel("ERROR")

    os.makedirs(DATAMART_DIR, exist_ok=True)

    print(f"Label definition in effect: bad=1 if overdue_amt > 0 at "
          f"installment_num == {MOB_THRESHOLD} (mob=6 fixed checkpoint).")

    for table_name in ["clickstream", "loan_daily", "financials", "attributes"]:
        run_bronze_silver(spark, table_name)

    silver_loan_daily_all = spark.read.parquet(os.path.join(SILVER_DIR, "loan_daily", "*.parquet"))
    silver_attributes_all = spark.read.parquet(os.path.join(SILVER_DIR, "attributes", "*.parquet"))
    silver_financials_all = spark.read.parquet(os.path.join(SILVER_DIR, "financials", "*.parquet"))
    silver_clickstream_all = spark.read.parquet(os.path.join(SILVER_DIR, "clickstream", "*.parquet"))

    process_gold_label_store(silver_loan_daily_all, GOLD_DIR)
    process_gold_feature_store(silver_attributes_all, silver_financials_all, silver_clickstream_all, GOLD_DIR)

    print("\nPipeline complete. See ./datamart for bronze/silver/gold tables.")
    spark.stop()


if __name__ == "__main__":
    main()

ModuleNotFoundError: No module named 'utils'